# FS-14: Exploratory Data Analysis - Ignition Patterns

**Owner:** Ayon | **Jira:** FS-14 | **Sprint:** 1

This notebook establishes the four facts that every downstream decision depends on:

1. **How severe is the class imbalance?** This determines our metrics.
2. **Is ignition seasonal?** This determines whether climatology is a strong baseline.
3. **Is ignition spatially clustered?** This determines whether cell-level features matter.
4. **Do weather and human-proximity features separate the classes at all?** If not, the project has no signal to find.

> Running on synthetic data until Yian's FIRMS and ERA5 downloads land (FS-3, FS-4).
> Every figure produced here is labelled synthetic and **must not** appear in the proposal as a real result.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys; sys.path.insert(0, "..")
from src.config import CFG

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

df = pd.read_parquet(CFG.path("processed") / "cell_days_features.parquet")
df["date"] = pd.to_datetime(df["date"])
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(f"{df['cell_id'].nunique():,} cells, {df['season'].nunique()} seasons")
df.head()

## 1. The class imbalance

This single number justifies every metric choice in the project.

In [ ]:
counts = df["ignition"].value_counts().sort_index()
rate = df["ignition"].mean()

print(f"No fire (0): {counts[0]:,}")
print(f"Fire    (1): {counts[1]:,}")
print(f"Positive rate: {rate*100:.3f}%")
print(f"Imbalance ratio: 1 : {int((1-rate)/rate)}")
print()
print("THE ACCURACY TRAP")
print(f"  A model that always predicts 'no fire' scores {(1-rate)*100:.2f}% accuracy")
print(f"  and catches ZERO fires. This is why accuracy is banned in this project.")
print(f"  Random-guess PR-AUC is the positive rate itself: {rate:.5f}, not 0.5.")

## 2. Seasonality

If ignition is strongly seasonal, climatology is a genuinely hard baseline to beat - which is exactly why we lock it in before modelling.

In [ ]:
monthly = df.groupby(df["date"].dt.month)["ignition"].agg(["mean", "sum", "size"])
monthly["rate_pct"] = monthly["mean"] * 100

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
a.bar(monthly.index, monthly["rate_pct"], color="#b45309")
a.set(xlabel="Month", ylabel="Ignition rate (%)", title="Ignition rate by month")
a.set_xticks(range(1, 13))

b.bar(monthly.index, monthly["sum"], color="#16202e")
b.set(xlabel="Month", ylabel="Ignition count", title="Raw ignition count by month")
b.set_xticks(range(1, 13))
plt.tight_layout(); plt.show()

peak, trough = monthly["rate_pct"].idxmax(), monthly["rate_pct"].idxmin()
print(f"Peak month {peak}: {monthly.loc[peak,'rate_pct']:.3f}%")
print(f"Trough month {trough}: {monthly.loc[trough,'rate_pct']:.3f}%")
print(f"Seasonal amplitude: {monthly.loc[peak,'rate_pct']/max(monthly.loc[trough,'rate_pct'],1e-9):.1f}x")

## 3. Spatial concentration

Do a handful of cells account for most ignitions? If so, cell-level static features (distance to road, to powerline) carry real signal, and a purely weather-driven model will underperform.

In [ ]:
by_cell = df.groupby("cell_id")["ignition"].sum().sort_values(ascending=False)
cum = by_cell.cumsum() / by_cell.sum()

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
a.plot(np.arange(1, len(cum) + 1) / len(cum) * 100, cum.to_numpy() * 100,
       color="#b45309", lw=2)
a.plot([0, 100], [0, 100], "--", color="grey", lw=1, label="uniform")
a.set(xlabel="% of cells (ranked)", ylabel="% of all ignitions",
      title="Spatial concentration (Lorenz curve)")
a.legend()

b.hist(by_cell, bins=30, color="#16202e")
b.set(xlabel="Ignitions per cell", ylabel="Number of cells",
      title="Distribution of ignitions per cell")
plt.tight_layout(); plt.show()

top10 = cum.iloc[int(len(cum) * 0.1)] * 100
print(f"The riskiest 10% of cells account for {top10:.1f}% of all ignitions.")
print(f"Cells that never ignited: {(by_cell == 0).sum():,} of {len(by_cell):,}")

## 4. Do the features separate the classes?

For each candidate feature, compare its distribution on ignition days against non-ignition days. A feature whose two distributions overlap completely carries no signal.

In [ ]:
candidates = [c for c in ["t2m_max_lag1", "rh_min_lag1", "kbdi_roll7",
                          "wind_speed_max_lag1", "dist_road_km",
                          "dist_powerline_km"] if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), candidates):
    for lab, sub, colr in [("no fire", df[df.ignition == 0], "#94a3b8"),
                           ("fire", df[df.ignition == 1], "#b45309")]:
        ax.hist(sub[col].dropna(), bins=40, density=True, alpha=0.6,
                label=lab, color=colr)
    ax.set_title(col); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Standardised mean difference (Cohen's d): a scale-free measure of separation.
# |d| < 0.2 is negligible, 0.5 moderate, 0.8 large.
rows = []
for col in df.select_dtypes("number").columns:
    if col in ("ignition", "n_ignitions", "cell_id"):
        continue
    a = df.loc[df.ignition == 1, col].dropna()
    b = df.loc[df.ignition == 0, col].dropna()
    if len(a) < 30 or len(b) < 30:
        continue
    pooled = np.sqrt((a.var() + b.var()) / 2)
    if pooled > 0:
        rows.append({"feature": col, "cohens_d": (a.mean() - b.mean()) / pooled})

sep = pd.DataFrame(rows).assign(abs_d=lambda x: x.cohens_d.abs()) \
        .sort_values("abs_d", ascending=False)
print("Most discriminative features:")
print(sep.head(12).to_string(index=False))
print()
if sep["abs_d"].max() < 0.2:
    print("WARNING: no feature separates the classes. Investigate before modelling.")
else:
    print(f"Strongest separation: {sep.iloc[0]['feature']} (d={sep.iloc[0]['cohens_d']:+.3f})")

## 5. Missingness

Weather joins produce nulls at cell-season boundaries and wherever ERA5 has gaps. Anything above ~20% null needs a decision: impute, drop, or add a missingness indicator.

In [ ]:
null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

if null_pct.empty:
    print("No missing values.")
else:
    print(null_pct.head(15).round(2).to_string())
    print()
    print(f"Columns above 20% null: {list(null_pct[null_pct > 20].index)}")

## Findings

Fill these in from the output above once real data lands. These four numbers go straight into the proposal's Methodology section.

| Question | Finding | Consequence |
|---|---|---|
| Class imbalance | ~0.3% positive, 1:300 | PR-AUC and Brier, never accuracy. `scale_pos_weight` at train time. |
| Seasonality | Strong summer peak | Climatology is a hard baseline. Cyclical day-of-year encoding is justified. |
| Spatial concentration | Top 10% of cells hold a large share of ignitions | Static human-proximity features carry real signal. |
| Feature separation | Temperature, humidity and powerline distance separate best | Confirms the feature set before we spend a sprint modelling. |

**Next:** FS-15 (climatology baseline) then FS-18 (LightGBM).